# 😎 MediaPipe 얼굴 랜드마크 AR 필터
## 내 얼굴에 안경 씌우기 — Google Colab 실습

이 실습에서는 **MediaPipe Face Landmarker**를 이용해 얼굴 랜드마크를 찾고, 눈 위치와 얼굴 기울기를 계산하여 **안경 AR 필터**를 합성합니다.

### 🎯 학습 목표
1. MediaPipe Face Landmarker 사용법 이해
2. 얼굴 랜드마크 좌표 이해
3. 눈의 중심점과 얼굴 기울기 계산
4. 투명 PNG의 Alpha Blending 이해
5. 얼굴 크기와 각도에 맞춘 AR 안경 합성

### 🧭 전체 흐름
`사진 입력 → Face Landmarker → 얼굴 랜드마크 → 눈 위치 계산 → 안경 크기/각도 계산 → AR 합성`

> 실행 방법: **런타임 → 모두 실행** 또는 각 셀을 위에서부터 순서대로 실행합니다.

## 1. 라이브러리 설치
MediaPipe 공식 Python API는 `mediapipe` 패키지로 설치합니다.

In [ ]:
!pip -q install mediapipe pillow

import cv2, math, os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import mediapipe as mp

print('MediaPipe version :', mp.__version__)
print('OpenCV version    :', cv2.__version__)

## 2. Face Landmarker 모델 다운로드
Google MediaPipe 모델 저장소에서 `.task` 모델 파일을 다운로드합니다.

In [ ]:
MODEL_PATH = '/content/face_landmarker.task'
MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
!wget -q -O "{MODEL_PATH}" "{MODEL_URL}"
print('모델 다운로드 완료:', os.path.exists(MODEL_PATH))
print('모델 크기(MB):', round(os.path.getsize(MODEL_PATH)/1024/1024, 2))

## 3. 얼굴 사진 업로드
정면에 가깝고 두 눈이 잘 보이는 JPG/PNG 사진을 권장합니다.

In [ ]:
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded.keys()))
image_bgr = cv2.imread(IMAGE_PATH)
if image_bgr is None:
    raise ValueError('이미지를 읽지 못했습니다. JPG/PNG 파일을 사용해 주세요.')
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(7,7)); plt.imshow(image_rgb); plt.axis('off'); plt.title('입력 이미지'); plt.show()
print('이미지 크기:', image_rgb.shape)

### 선택 실습 — Colab 웹캠으로 직접 촬영
이미지를 업로드했다면 이 셀은 건너뛰어도 됩니다.

In [ ]:
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Javascript, display

def take_photo(filename='/content/webcam_photo.jpg', quality=0.9):
    js = Javascript(r"""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = '📷 사진 촬영';
      button.style.fontSize = '18px'; button.style.padding = '10px 18px';
      const video = document.createElement('video');
      video.style.display = 'block'; video.style.width = '640px';
      video.setAttribute('autoplay',''); video.setAttribute('playsinline','');
      div.appendChild(video); div.appendChild(button); document.body.appendChild(div);
      const stream = await navigator.mediaDevices.getUserMedia({video:true});
      video.srcObject = stream; await video.play();
      await new Promise(resolve => button.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth; canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video,0,0);
      stream.getVideoTracks()[0].stop(); div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    """)
    display(js)
    data = eval_js(f'takePhoto({quality})')
    binary = b64decode(data.split(',')[1])
    with open(filename,'wb') as f: f.write(binary)
    return filename

# 사용하려면 아래 두 줄의 주석을 해제하세요.
# IMAGE_PATH = take_photo()
# print('촬영 완료:', IMAGE_PATH)

## 4. MediaPipe Face Landmarker 생성
한 장의 사진을 처리하므로 `IMAGE` 모드를 사용합니다.

In [ ]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5
)
print('Face Landmarker 옵션 준비 완료')

## 5. 얼굴 랜드마크 검출
Face Landmarker는 얼굴마다 고밀도 정규화 `(x, y, z)` 랜드마크를 반환합니다.

In [ ]:
image_bgr = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
with FaceLandmarker.create_from_options(options) as landmarker:
    result = landmarker.detect(mp_image)
if len(result.face_landmarks) == 0:
    raise RuntimeError('얼굴을 찾지 못했습니다. 정면 얼굴 사진으로 다시 시도해 주세요.')
landmarks = result.face_landmarks[0]
print('✅ 얼굴 검출 성공')
print('랜드마크 개수:', len(landmarks))
print('첫 번째 랜드마크:', landmarks[0])

## 6. 얼굴 랜드마크 시각화
정규화 좌표를 픽셀 좌표로 바꾸어 얼굴 위에 점을 표시합니다.

In [ ]:
h, w = image_rgb.shape[:2]
landmark_img = image_rgb.copy()
for lm in landmarks:
    x, y = int(lm.x*w), int(lm.y*h)
    cv2.circle(landmark_img,(x,y),1,(0,255,0),-1)
KEY_INDICES = [33,133,362,263]
for idx in KEY_INDICES:
    lm = landmarks[idx]; x, y = int(lm.x*w), int(lm.y*h)
    cv2.circle(landmark_img,(x,y),6,(255,0,0),-1)
    cv2.putText(landmark_img,str(idx),(x+7,y-7),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,0),2)
plt.figure(figsize=(9,9)); plt.imshow(landmark_img); plt.axis('off'); plt.title(f'Face Landmarks ({len(landmarks)} points)'); plt.show()

## 7. 두 눈의 중심점·거리·기울기 계산
안경 중심, 크기, 회전각을 계산합니다.

In [ ]:
def to_pixel(lm, width, height):
    return np.array([lm.x*width, lm.y*height], dtype=np.float32)

eye_a = (to_pixel(landmarks[33],w,h)+to_pixel(landmarks[133],w,h))/2
eye_b = (to_pixel(landmarks[362],w,h)+to_pixel(landmarks[263],w,h))/2
left_on_image, right_on_image = (eye_a,eye_b) if eye_a[0] < eye_b[0] else (eye_b,eye_a)
eye_mid = (left_on_image+right_on_image)/2
eye_distance = np.linalg.norm(right_on_image-left_on_image)
dx, dy = right_on_image-left_on_image
angle_deg = math.degrees(math.atan2(dy,dx))
print('왼쪽 눈 중심:', np.round(left_on_image,1))
print('오른쪽 눈 중심:', np.round(right_on_image,1))
print('두 눈 사이 거리(px):', round(float(eye_distance),1))
print('눈 기울기(도):', round(float(angle_deg),2))

## 8. 투명 안경 PNG 직접 만들기
외부 이미지 없이 Pillow로 투명 안경을 생성합니다.

In [ ]:
def create_glasses_png(save_path='/content/glasses.png', frame_color=(25,25,30,255), lens_color=(130,190,255,45)):
    W,H=1000,360
    img=Image.new('RGBA',(W,H),(0,0,0,0)); d=ImageDraw.Draw(img)
    d.rounded_rectangle((90,85,410,295),radius=85,fill=lens_color,outline=frame_color,width=28)
    d.rounded_rectangle((590,85,910,295),radius=85,fill=lens_color,outline=frame_color,width=28)
    d.arc((395,125,605,245),start=190,end=350,fill=frame_color,width=24)
    d.line((90,135,5,90),fill=frame_color,width=24)
    d.line((910,135,995,90),fill=frame_color,width=24)
    img.save(save_path); return save_path
GLASSES_PATH=create_glasses_png()
plt.figure(figsize=(10,4)); plt.imshow(Image.open(GLASSES_PATH)); plt.axis('off'); plt.title('실습용 투명 안경 PNG'); plt.show()

## 9. AR 합성 함수
눈 사이 거리로 크기를 정하고, 눈의 기울기로 회전한 뒤 Alpha Blending으로 합성합니다.

In [ ]:
def alpha_overlay(base_rgb, overlay_rgba, center_xy):
    base=base_rgb.copy(); oh,ow=overlay_rgba.shape[:2]; cx,cy=map(int,center_xy)
    x1,y1=cx-ow//2,cy-oh//2; x2,y2=x1+ow,y1+oh
    bx1,by1=max(0,x1),max(0,y1); bx2,by2=min(base.shape[1],x2),min(base.shape[0],y2)
    if bx1>=bx2 or by1>=by2: return base
    ox1,oy1=bx1-x1,by1-y1; ox2,oy2=ox1+(bx2-bx1),oy1+(by2-by1)
    crop=overlay_rgba[oy1:oy2,ox1:ox2]
    alpha=crop[:,:,3:4].astype(np.float32)/255.0
    rgb=crop[:,:,:3].astype(np.float32)
    region=base[by1:by2,bx1:bx2].astype(np.float32)
    base[by1:by2,bx1:bx2]=(alpha*rgb+(1-alpha)*region).astype(np.uint8)
    return base

def put_glasses_on_face(image_rgb, landmarks, glasses_path='/content/glasses.png', size_factor=2.45, vertical_offset=0.08):
    h,w=image_rgb.shape[:2]
    def p(idx):
        lm=landmarks[idx]; return np.array([lm.x*w,lm.y*h],dtype=np.float32)
    e1=(p(33)+p(133))/2; e2=(p(362)+p(263))/2
    left,right=(e1,e2) if e1[0]<e2[0] else (e2,e1)
    mid=(left+right)/2; dist=float(np.linalg.norm(right-left))
    dx,dy=right-left; angle=math.degrees(math.atan2(dy,dx))
    glasses=Image.open(glasses_path).convert('RGBA')
    target_w=max(80,int(dist*size_factor)); scale=target_w/glasses.width
    target_h=max(30,int(glasses.height*scale))
    glasses=glasses.resize((target_w,target_h),Image.Resampling.LANCZOS)
    glasses=glasses.rotate(-angle,resample=Image.Resampling.BICUBIC,expand=True)
    center=mid.copy(); center[1]+=dist*vertical_offset
    result=alpha_overlay(image_rgb,np.array(glasses),center)
    return result, {'eye_distance':dist,'angle':angle,'glasses_width':target_w}

## 10. 😎 AR 안경 필터 실행

In [ ]:
ar_result, info = put_glasses_on_face(image_rgb, landmarks, GLASSES_PATH, size_factor=2.45, vertical_offset=0.08)
plt.figure(figsize=(10,10)); plt.imshow(ar_result); plt.axis('off'); plt.title('😎 MediaPipe AR Glasses Filter'); plt.show()
print(info)

## 11. 결과 저장 및 다운로드

In [ ]:
OUTPUT_PATH='/content/AR_glasses_result.png'
Image.fromarray(ar_result).save(OUTPUT_PATH)
from google.colab import files
files.download(OUTPUT_PATH)
print('저장 완료:', OUTPUT_PATH)

# 🧪 실습 미션 1 — 안경 크기 바꾸기
`size_factor`를 `2.1`, `2.45`, `2.8` 등으로 바꿔 비교하세요.

In [ ]:
MY_SIZE=2.70
test_result,_=put_glasses_on_face(image_rgb,landmarks,GLASSES_PATH,size_factor=MY_SIZE,vertical_offset=0.08)
plt.figure(figsize=(8,8)); plt.imshow(test_result); plt.axis('off'); plt.title(f'size_factor = {MY_SIZE}'); plt.show()

# 🧪 실습 미션 2 — 안경 프레임 색 바꾸기

In [ ]:
CUSTOM_GLASSES='/content/custom_glasses.png'
create_glasses_png(CUSTOM_GLASSES, frame_color=(230,40,40,255), lens_color=(255,170,170,45))
custom_result,_=put_glasses_on_face(image_rgb,landmarks,CUSTOM_GLASSES,size_factor=2.45,vertical_offset=0.08)
plt.figure(figsize=(8,8)); plt.imshow(custom_result); plt.axis('off'); plt.title('나만의 안경 프레임'); plt.show()

# 🚀 도전 과제

### 초급
1. 안경 색을 바꿔 보세요.
2. `vertical_offset`을 조절해 위치를 바꿔 보세요.
3. 랜드마크 1, 10, 152, 234, 454를 표시해 보세요.

### 중급
4. 코 랜드마크로 빨간 코 필터를 추가하세요.
5. 입 주변 랜드마크로 콧수염 필터를 만드세요.
6. `num_faces`를 늘려 여러 얼굴을 처리하세요.

### 심화
7. 동영상을 프레임 단위로 처리하세요.
8. `VIDEO` 또는 `LIVE_STREAM` 모드로 바꿔 실시간 필터를 구현하세요.
9. Face Blendshapes로 웃음/눈 깜빡임에 반응하는 필터를 만드세요.
10. 모자, 마스크, 캐릭터 귀 등 다른 AR 스티커로 확장하세요.

# 📌 핵심 정리

- 픽셀 좌표: `pixel_x = x × width`, `pixel_y = y × height`
- 눈 사이 거리: `√((x₂-x₁)² + (y₂-y₁)²)`
- 회전각: `atan2(y₂-y₁, x₂-x₁)`
- Alpha Blending: `결과 = α×안경 + (1-α)×원본`

즉, **랜드마크가 AR 오브젝트의 위치·크기·각도를 결정**합니다.

## 🔗 공식 참고 자료
- MediaPipe Face Landmarker Python: https://developers.google.com/edge/mediapipe/solutions/vision/face_landmarker/python
- MediaPipe Face Landmarker API: https://ai.google.dev/edge/api/mediapipe/python/mp/tasks/vision/FaceLandmarker
